# Week 2 Day 1 — Train / Val / Test Split
**Jul 8, 2026**

Day 6 split into train/val. Today: a proper **three-way** split, and a real problem that shows up on imbalanced data (like default rates in credit data, where positives might be 5-10% of rows) — plain random splitting can accidentally give your val or test set a noticeably different class balance than train, purely by chance. You'll see that happen, then fix it with a stratified split.

Scaffold as usual: TODO stubs, hints not formulas, self-check cells.

## Part 1: Imbalanced data, and the problem with naive splitting

Given. 1000 samples, only 10% positive — deliberately imbalanced. Then a plain `random_split` into 70/15/15, with the positive rate printed per split.

In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split

torch.manual_seed(0)
n_samples, n_features = 1000, 4
n_pos = 100

y = torch.cat([torch.ones(n_pos), torch.zeros(n_samples - n_pos)])
X = torch.randn(n_samples, n_features)
ds = TensorDataset(X, y)

gen = torch.Generator().manual_seed(1)
train_s, val_s, test_s = random_split(ds, [0.7, 0.15, 0.15], generator=gen)

for name, s in [("train", train_s), ("val", val_s), ("test", test_s)]:
    print(name, len(s), "positive rate:", y[s.indices].mean().item())

print("\noverall positive rate:", y.mean().item())

train 700 positive rate: 0.1071428582072258
val 150 positive rate: 0.07333333045244217
test 150 positive rate: 0.09333333373069763

overall positive rate: 0.10000000149011612


Notice the per-split positive rates aren't exactly 10% — plain random chance drifted them apart. On a small or heavily imbalanced dataset this can get much worse (a val set with almost no positive examples is a real failure mode, not a hypothetical one).

## Part 2: Stratified split

TODO: implement `stratified_split(y, ratios, seed)` returning a list of index tensors (one per ratio), where **each split preserves the overall class balance**.

The core idea, not the code: don't shuffle and split the whole dataset at once. Instead, handle each class *separately* — shuffle that class's indices, cut them according to `ratios`, then combine the corresponding pieces across classes into each final split. That way every split gets the same proportion of each class as the full dataset, by construction rather than by luck.

Things to watch:
- `y.unique()` gives you the distinct classes to loop over.
- Getting the indices belonging to one class: think about what comparison produces a boolean mask, and how to turn that into actual index positions (`.nonzero()` is the relevant tool).
- Rounding: `ratios` won't divide a class's count evenly in general — decide how you handle the leftover few samples (e.g. via cumulative rounding) so every sample ends up in exactly one split, with none dropped or duplicated.

In [2]:
def stratified_split(y, ratios, seed=0):
    assert abs(sum(ratios) - 1.0) < 1e-6
    # TODO: return a list of 1-D index tensors, one per entry in `ratios`,
    classes = torch.unique(y)
    split_indices = [[] for _ in ratios]
    
    #local random generator for reproducibility
    generator = torch.Generator().manual_seed(seed)

    for c in classes:
        idx = (y == c).nonzero(as_tuple=True)[0]
        
        #shuffle indices for this class
        idx =idx[torch.randperm(len(idx), generator=generator)]
        
        # Split the indices
        start = 0
        for i, ratio in enumerate(ratios):
            if i == len(ratios) - 1:
                end = len(idx)
            else:
                end = start + int(len(idx) * ratio)
                
            split_indices[i].append(idx[start:end])
            start = end
            
        # combine the indices for each split
    split_indices = [torch.cat(indices) for indices in split_indices]
        
        #shuffle each final splits
    split_indices = [
        idx[torch.randperm(len(idx), generator=generator)] for idx in split_indices
        ]


    return split_indices


train_idx, val_idx, test_idx = stratified_split(y, [0.7, 0.15, 0.15], seed=1)

# self-check: sizes add up, no samples dropped or duplicated
all_idx = torch.cat([train_idx, val_idx, test_idx])
assert len(all_idx) == n_samples
assert len(all_idx.unique()) == n_samples, "a sample appears in more than one split, or was dropped"

# self-check: class balance preserved (within a small tolerance)
overall_rate = y.mean().item()
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    rate = y[idx].mean().item()
    print(name, len(idx), "positive rate:", rate)
    assert abs(rate - overall_rate) < 0.01, f"{name} split drifted from overall balance"

print("\nstratified split OK -- balance preserved in every split")

train 700 positive rate: 0.10000000149011612
val 150 positive rate: 0.10000000149011612
test 150 positive rate: 0.10000000149011612

stratified split OK -- balance preserved in every split


## Part 3: Wire it into `DataLoader`s

TODO: using `torch.utils.data.Subset(ds, indices)` with the index tensors from Part 2, build three `DataLoader`s. Only `train_loader` should shuffle.

In [3]:
from torch.utils.data import Subset

# TODO: train_loader = DataLoader(Subset(ds, train_idx), ...)
train_loader = DataLoader(Subset(ds, train_idx), batch_size=32, shuffle=True)
# TODO: val_loader = DataLoader(Subset(ds, val_idx), ...)
val_loader = DataLoader(Subset(ds, val_idx), batch_size=32, shuffle=False)
# TODO: test_loader = DataLoader(Subset(ds, test_idx), ...)
test_loader = DataLoader(Subset(ds, test_idx), batch_size=32, shuffle=False)

xb, yb = next(iter(train_loader))
print("train batch:", xb.shape, yb.shape)

train batch: torch.Size([32, 4]) torch.Size([32])


## A rule worth internalizing now, before it matters

`val` gets checked repeatedly during development — every epoch, every hyperparameter you try. `test` gets touched **once**, at the very end, after every decision about the model is already final. The moment you tune anything based on a test-set number, it quietly becomes a second validation set, and you've lost your only unbiased estimate of real performance. There's no code to enforce this — it's a discipline you keep on yourself.

## Try yourself

1. Push `n_pos` down to something much smaller (e.g. 20 out of 1000) and rerun Part 1's naive split several times with different seeds. How bad does the drift get, and does any split ever end up with zero positives?
2. Extend `stratified_split` to work with more than two classes without changes (it should already, if you looped over `y.unique()` generically — verify it on a 3-class synthetic `y`).
3. Implement k-fold cross-validation as a generalization: instead of one fixed train/val split, produce `k` different (train_idx, val_idx) pairs where every sample appears in exactly one val fold.

## Bonus: stratified k-fold

TODO: implement `k_fold_split(y, k, seed)` returning a list of `k` `(train_idx, val_idx)` pairs, where every sample appears in exactly one val fold across the whole list — the generalization of `stratified_split` to k-fold cross-validation.

Reuse, don't rebuild: the structure is almost identical to `stratified_split` above. Loop over each class, shuffle its indices the same way, but instead of cutting at arbitrary `ratios`, cut into `k` equal-ish chunks (`torch.chunk` does this in one call). Collect fold `i`'s chunk from every class the same way `stratified_split` collected each split's chunk, then `torch.cat` per fold. Finally, turn the `k` folds into `k` pairs: for each `i`, `val_idx = folds[i]` and `train_idx` is every other fold concatenated together.

In [4]:
import torch


def k_fold_split(y, k, seed=0):
    assert k >= 2
    assert y.ndim == 1

    classes = torch.unique(y)
    generator = torch.Generator().manual_seed(seed)

    # One list for each validation fold
    val_folds = [[] for _ in range(k)]

    for c in classes:
        # Find all indices belonging to class c
        idx = (y == c).nonzero(as_tuple=True)[0]

        # Each class should have enough samples for k folds
        assert len(idx) >= k, (
            f"Class {c.item()} has only {len(idx)} samples, "
            f"which is fewer than k={k}"
        )

        # Shuffle this class's indices
        idx = idx[
            torch.randperm(len(idx), generator=generator)
        ]

        # Divide this class approximately evenly among k folds
        class_folds = torch.tensor_split(idx, k)

        # Add one class-specific piece to each validation fold
        for fold_i in range(k):
            val_folds[fold_i].append(class_folds[fold_i])

    # Combine the different classes inside each validation fold
    val_folds = [
        torch.cat(class_parts)
        for class_parts in val_folds
    ]

    # Shuffle each validation fold
    val_folds = [
        fold[torch.randperm(len(fold), generator=generator)]
        for fold in val_folds
    ]

    pairs = []

    for fold_i in range(k):
        val_idx = val_folds[fold_i]

        # Training data consists of all other validation folds
        train_idx = torch.cat([
            val_folds[j]
            for j in range(k)
            if j != fold_i
        ])

        # Optionally shuffle the training indices
        train_idx = train_idx[
            torch.randperm(len(train_idx), generator=generator)
        ]

        pairs.append((train_idx, val_idx))

    return pairs